# Farm boundaries from PlanetScope: preprocess, train, compare, map (Kaggle, end to end)

One **Save & Run All** does everything:

1. **Preprocess** the PlanetScope scenes and your Google Earth Pro farm polygons: interior / boundary / instance /
   valid masks on each scene's exact grid, a geographic train/val/test split, 512 px patches, normalisation,
   automated pre-training checks.
2. **Train** several experiments (encoder, boundary width, training labels), one after the other.
3. **Compare** them fairly: every model is scored on the *same* validation farms (your hand-traced
   `Digitized` farms), at the level that matters, **individual farms** (object F1, IoU ≥ 0.5), with
   post-processing thresholds tuned per model on validation.
4. **Report the winner on the test blocks** (never used for any choice), and
5. **map every scene** with it: polygons as KMZ (Google Earth Pro), GeoJSON, Shapefile and GeoPackage (QGIS).

**Inputs**
* *Add Input → Your Work → Datasets*: one **private** dataset holding the Planet order folder(s) downloaded
  with `python -m src planet-download` (the `*_SR_8b_clip.tif` scenes and their `*_udm2_clip.tif` masks).
  Planet imagery is licensed for research: keep the dataset private.
* The code and your farm polygons (`farm_boundary_detection/data/labels/`) come from the GitHub repository.

**Settings:** *Accelerator → GPU T4 x2 or P100*, *Internet → On*, then **Save Version → Save & Run All**.
A full run is roughly 5–8 h (12 h limit). It resumes: attach this notebook's previous output as an input and
finished steps are skipped, interrupted training continues from `latest_checkpoint.pth`.
Set `QUICK_TEST = True` first for a ~15 min dry run of the whole notebook.

## 1. Settings

In [ ]:
from pathlib import Path

DATASET_NAME = "laikipia-planetscope"   # the private Kaggle dataset with the Planet order folder(s)
REPO = "https://github.com/mangaorphy/Farm_Digitisation.git"
BRANCH = "main"
PROJECT_SUBDIR = "farm_boundary_detection"

QUICK_TEST = False        # True = tiny end-to-end check (few patches, 2 epochs)
EPOCHS = 80               # upper limit; early stopping ends each experiment when validation stops improving
PATIENCE = 12
BATCH = 8                 # 512 px patches; lower to 4 if CUDA runs out of memory
MAP_ALL_SCENES = True     # map every scene with the best model at the end

# Preprocessing variants: experiments that need different target masks get their own patches.
# All variants share ONE geographic split (built from the first variant), so no test block is ever trained on.
PREP = {
    "b3_all":    ["masks.boundary_width_m=3.0"],                       # 1-pixel boundaries, all QA-OK polygons
    "b6_all":    ["masks.boundary_width_m=6.0"],                       # 2-pixel boundaries
    "b3_traced": ["masks.boundary_width_m=3.0",                        # hand-traced farms only: the scheme layer
                  "labels.keep={QA_FLAG: [OK], SOURCE: [Digitized]}"], # (plans, maybe misregistered) is ignored
}
# Every model is compared on the validation patches of this variant (trusted hand-traced farms only).
REFERENCE_PREP = "b3_traced"

# experiment -> (preprocessing variant, model/training overrides). Order = simplest first (wins ties).
EXPERIMENTS = {
    "E1_r34_b3":        ("b3_all",    ["model.encoder=resnet34"]),
    "E2_r34_b6":        ("b6_all",    ["model.encoder=resnet34"]),
    "E3_r34_b3_traced": ("b3_traced", ["model.encoder=resnet34"]),
    "E4_r50_b3":        ("b3_all",    ["model.encoder=resnet50", "train.batch_size=6"]),
}

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

## 2. Install and get the code

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rasterio", "geopandas", "pyogrio",
                "scikit-image"], check=True)
code_dir = WORK / "Farm_Digitisation"
if not code_dir.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, str(code_dir)], check=True)
else:
    subprocess.run(["git", "-C", str(code_dir), "pull", "--ff-only"], check=False)
PROJECT = code_dir / PROJECT_SUBDIR
sys.path.insert(0, str(PROJECT))
os.chdir(PROJECT)

import torch, src
print("farm_boundary_detection", src.__version__, "| torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator!")

## 3. Find the imagery

In [ ]:
def find_dataset(name):
    hits = sorted((p for p in INPUT.rglob("*") if p.is_dir() and p.name == name), key=lambda p: len(p.parts))
    return hits[0] if hits else None

IMAGERY = find_dataset(DATASET_NAME)
assert IMAGERY, f"dataset {DATASET_NAME!r} not attached (Add Input). Inputs: {[p.name for p in INPUT.iterdir()]}"
tifs = sorted(p for p in IMAGERY.rglob("*.tif") if "udm" not in p.name.lower())
udm = sorted(p for p in IMAGERY.rglob("*.tif") if "udm2" in p.name.lower())
print(f"{IMAGERY}: {len(tifs)} scenes, {len(udm)} UDM2 masks")
for p in tifs:
    print("  ", p.relative_to(IMAGERY))
assert tifs, "no PlanetScope GeoTIFFs found in the dataset"

## 4. Configuration helper

In [ ]:
import json, shutil, logging
import pandas as pd
from src.utils.helpers import load_config, setup_logging

setup_logging(WORK / "logs", "INFO", "kaggle")
log = logging.getLogger("notebook")

QUICK = ["train.max_train_patches=48", "patches.background_keep_fraction=0.05", "train.epochs=2",
         "evaluation.n_visuals=4"] if QUICK_TEST else []

def make_cfg(prep, exp=None, extra=()):
    exp_dir = WORK / "experiments" / (exp or "_none")
    over = [f"paths.imagery_dir={IMAGERY}", f"paths.processed_dir={WORK / 'prep' / prep}",
            f"paths.models_dir={exp_dir / 'models'}", f"paths.outputs_dir={exp_dir / 'outputs'}",
            f"paths.logs_dir={WORK / 'logs'}", "train.num_workers=4", f"train.batch_size={BATCH}",
            f"train.epochs={EPOCHS}", f"train.patience={PATIENCE}", *PREP[prep], *extra, *QUICK]
    return load_config(PROJECT / "configs" / "config.yaml", over)

## 5. Preprocess every variant

Masks → validation figures → geographic split → patches → normalisation → pre-training checks
(dimensions, CRS, overlap, boundary presence, interior/boundary alignment, disjoint splits, nodata excluded).
A failed check stops the notebook with the reason (details in `prep/<variant>/metadata/validation_report.json`).

In [ ]:
from src.pipeline import prepare

first = next(iter(PREP))
for prep in PREP:
    cfg = make_cfg(prep)
    meta = Path(cfg.paths.processed_dir) / "metadata"
    if (meta / "patches.csv").exists() and (meta / "validation_report.json").exists():
        print(f"{prep}: already prepared"); continue
    if prep != first:   # share the split of the first variant
        meta.mkdir(parents=True, exist_ok=True)
        shutil.copy(WORK / "prep" / first / "metadata" / "split_blocks.json", meta / "split_blocks.json")
    print(f"\n========== preprocessing {prep} ==========")
    prepare(cfg)

for prep in PREP:
    t = pd.read_csv(WORK / "prep" / prep / "metadata" / "patches.csv")
    print(prep, t.groupby("split").agg(patches=("patch_id", "size"), background=("is_background", "sum"),
                                        farms=("n_farms", "sum")).to_dict("index"))
rep = json.loads((WORK / "prep" / first / "metadata" / "preprocessing_report.json").read_text())
print({k: rep[k] for k in ("source_polygons", "valid_polygons", "repaired", "removed_exact_duplicates",
                           "removed_near_duplicates", "train_polygons", "ignored_polygons")})
pd.DataFrame(rep["rasters"])[["scene_id", "polygons_intersecting", "polygons_outside", "size", "bands", "udm2"]]

Mask validation (image, interior mask, boundary mask, boundaries over the image) and the split map:

In [ ]:
from IPython.display import Image, display
for f in sorted((WORK / "prep" / first / "visualizations").glob("*.png"))[:8]:
    print(f.name); display(Image(filename=str(f), width=1400))

## 6. Train every experiment

In [ ]:
from src.training.trainer import train
from src.utils.visualization import plot_history

best_pt = {}
for exp, (prep, extra) in EXPERIMENTS.items():
    cfg = make_cfg(prep, exp, extra)
    mdir = Path(cfg.paths.models_dir); mdir.mkdir(parents=True, exist_ok=True)
    # continue from a previous version's output if it is attached as an input
    prev = sorted(INPUT.rglob(f"experiments/{exp}/models/latest_checkpoint.pth"))
    if prev and not (mdir / "latest_checkpoint.pth").exists():
        for f in prev[0].parent.glob("*"):
            shutil.copy(f, mdir / f.name)
        print(f"{exp}: resuming from {prev[0]}")
    print(f"\n========== {exp}: {prep} {extra} ==========")
    best_pt[exp] = train(cfg)
    plot_history(mdir / "training_history.csv", mdir / "training_history.png")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for exp in EXPERIMENTS:
    h = pd.read_csv(WORK / "experiments" / exp / "models" / "training_history.csv")
    ax[0].plot(h.epoch, h.val_loss, label=exp); ax[1].plot(h.epoch, h.interior_iou, label=exp)
    ax[2].plot(h.epoch, h.boundary_f1, label=exp)
for a, t in zip(ax, ["validation loss", "interior IoU (val)", "boundary F1, 1 px tolerance (val)"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.show()

## 7. Compare on the common validation reference

Each model predicts the validation patches of `REFERENCE_PREP` once; its post-processing thresholds
(extent, seed, minimum seed size) are tuned on those patches; the model with the best **object F1**
(farms matched one-to-one at IoU ≥ 0.5) wins. Ties within 0.01 go to the simpler (earlier) experiment.

In [ ]:
from src.training.evaluation import predict_patches, tune_postprocess, evaluate

ref_cfg = make_cfg(REFERENCE_PREP)
rows, tuned = {}, {}
for exp in EXPERIMENTS:
    cfg = make_cfg(REFERENCE_PREP, exp, EXPERIMENTS[exp][1])
    cache = predict_patches(cfg, best_pt[exp], "val")
    pc, grid = tune_postprocess(cfg, cache)
    tuned[exp] = pc
    res = evaluate(cfg, best_pt[exp], "val", Path(cfg.paths.outputs_dir) / "evaluation" / "val_reference",
                   postprocess=pc, cache=cache)
    rows[exp] = {k: round(res[k], 4) if isinstance(res[k], float) else res[k] for k in
                 ("object_f1", "object_precision", "object_recall", "mean_matched_iou", "merged_predictions",
                  "split_farms", "interior_iou", "boundary_f1", "boundary_iou", "farms_true")}
    rows[exp].update({f"pp_{k}": pc[k] for k in ("interior_threshold", "seed_threshold", "min_seed_px")})
    del cache

comparison = pd.DataFrame(rows).T
comparison.to_csv(WORK / "comparison_val.csv")
top = comparison.object_f1.astype(float).max()
BEST = next(e for e in EXPERIMENTS if float(comparison.loc[e, "object_f1"]) >= top - 0.01)
print(f"BEST = {BEST} (val object F1 {float(comparison.loc[BEST, 'object_f1']):.3f}; top {top:.3f})")
comparison

## 8. Test the winner (held-out blocks, reported once)

In [ ]:
best_dir = WORK / "best"; best_dir.mkdir(exist_ok=True)
cfg_ref = make_cfg(REFERENCE_PREP, BEST, EXPERIMENTS[BEST][1])
test_ref = evaluate(cfg_ref, best_pt[BEST], "test", best_dir / "test_reference", postprocess=tuned[BEST])
cfg_own = make_cfg(EXPERIMENTS[BEST][0], BEST, EXPERIMENTS[BEST][1])
test_own = evaluate(cfg_own, best_pt[BEST], "test", best_dir / "test_own_labels", postprocess=tuned[BEST])

shutil.copy(best_pt[BEST], best_dir / "best_model.pth")
summary = {"best_experiment": BEST, "settings": EXPERIMENTS[BEST], "postprocess": dict(tuned[BEST]),
           "val_reference": rows[BEST], "test_reference": test_ref, "test_own_labels": test_own}
(best_dir / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
keys = ["object_f1", "object_precision", "object_recall", "mean_matched_iou", "merged_predictions", "split_farms",
        "interior_iou", "interior_dice", "interior_precision", "interior_recall", "boundary_iou", "boundary_f1",
        "boundary_precision", "boundary_recall", "farms_true"]
pd.DataFrame({"test (traced farms)": {k: test_ref[k] for k in keys},
              "test (training labels)": {k: test_own[k] for k in keys}})

In [ ]:
for f in sorted((best_dir / "test_reference" / "visuals").glob("*.png"))[:6]:
    display(Image(filename=str(f), width=1500))

## 9. Map every scene with the best model

Probabilities per scene → farm polygons (watershed seeded inside farms, split along the predicted boundary)
→ one merged layer (a farm seen in two overlapping scenes is kept once) → KMZ / GeoJSON / Shapefile / GPKG.

In [ ]:
from src.inference.predictor import Predictor
from src.gis.polygon_reconstruction import polygonize, export_polygons, merge_scenes

if MAP_ALL_SCENES:
    cfg_map = make_cfg(EXPERIMENTS[BEST][0], BEST, EXPERIMENTS[BEST][1])
    cfg_map.paths.outputs_dir = best_dir / "map"
    cfg_map.postprocess = tuned[BEST]
    predictor = Predictor(best_pt[BEST], cfg_map)
    frames = []
    for img in tifs:
        prob = predictor.predict_raster(img)
        farms = polygonize(cfg_map, prob)
        export_polygons(farms, cfg_map.paths.outputs_dir, prob.stem.replace("_prob", "") + "_farms")
        frames.append(farms)
    allf = merge_scenes(frames)
    paths = export_polygons(allf, cfg_map.paths.outputs_dir, "all_farms")
    print(f"{len(allf)} farms, {allf.area_ha.sum():.0f} ha, {int((allf.score < 0.6).sum())} low-confidence (orange in the KMZ)")
    print({k: str(v) for k, v in paths.items()})

In [ ]:
# quick look: predicted farms (yellow) over the image, for the densest 3 x 3 km of the first scene
import numpy as np, rasterio
from rasterio.windows import from_bounds
from src.utils.visualization import stretch
if MAP_ALL_SCENES and len(frames[0]):
    f0 = frames[0]
    c = f0.geometry.centroid
    cell = (np.floor(c.x / 3000) * 3000).astype(int).astype(str) + "_" + (np.floor(c.y / 3000) * 3000).astype(int).astype(str)
    x0, y0 = map(float, cell.value_counts().index[0].split("_"))
    b = (x0, y0, x0 + 3000, y0 + 3000)
    with rasterio.open(tifs[0]) as s:
        rgb = s.read([6, 4, 2] if s.count == 8 else [3, 2, 1], window=from_bounds(*b, s.transform))
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(stretch(rgb), extent=(b[0], b[2], b[1], b[3]))
    f0.cx[b[0]:b[2], b[1]:b[3]].boundary.plot(ax=ax, color="yellow", linewidth=0.8)
    ax.set_xlim(b[0], b[2]); ax.set_ylim(b[1], b[3]); ax.set_title(f"{BEST}: predicted farms"); ax.axis("off")

## 10. Download

From the notebook *Output* (folder `best/`):

* `map/kmz/all_farms.kmz`: every farm for **Google Earth Pro** (green = confident, orange = score < 0.6: review first);
  `map/geojson/`, `map/shapefiles/`, `map/gpkg/` for **QGIS**; one file per scene as well;
* `best_model.pth`: the winning model (with its normalisation and settings) and `summary.json`: its settings,
  tuned post-processing thresholds, validation and test scores;
* `test_reference/visuals/`: predictions vs your traced farms on test patches;

and `comparison_val.csv` (all experiments), `experiments/<E>/models/` (every model, training history).